# 04 — Train: SARIMAX

Fits one fixed-order SARIMAX model with a deliberately minimal daily seasonal AR term using the six calendar Fourier terms as exogenous regressors for the configured target station and evaluates it once on the test feature artifact. A local ADF/ACF/PACF diagnostic on the train water-level series found negligible daily autocorrelation after detrending, so the seasonal order tests for a weak daily effect rather than assuming a large one. Weather and water-level lag/rolling features are excluded from exog because multi-step forecasting needs their values at every future step, and future weather is never known in advance.

**Inputs:** train-derived and test-derived feature artifacts  
**Outputs:** in-notebook prediction preview and test metrics only

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports dependencies and fixes the notebook's fixed SARIMAX order, calendar exogenous regressor columns, artifact paths, preview row count, and feature/target column lists.

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

from src.config import FORECAST_HORIZON_HOURS, TARGET_STATION_ID
from src.feature_engineering import (
    DEFAULT_FEATURE_CONFIG,
    feature_column_names,
    target_column_names,
)

PROCESSED_DIR = Path("data/processed")
PREDICTION_PREVIEW_ROWS = 5
FEATURE_COLUMNS = list(feature_column_names())
TARGET_COLUMNS = list(target_column_names())
CALENDAR_EXOG_COLUMNS = [
    "utc_hour_sin",
    "utc_hour_cos",
    "utc_day_of_week_sin",
    "utc_day_of_week_cos",
    "utc_day_of_year_sin",
    "utc_day_of_year_cos",
]
CALENDAR_TIMEZONE = DEFAULT_FEATURE_CONFIG.calendar_timezone
TRAIN_WATER_LEVEL_INTERPOLATION = {
    "method": "linear",
    "limit_area": "inside",
    "scope": "training input passed to SARIMAX only",
}
SARIMAX_ORDER = (1, 1, 1)
SARIMAX_SEASONAL_ORDER = (1, 0, 0, 24)

## Shared evaluation cohort

SARIMAX uses water level plus the six calendar Fourier terms as exogenous regressors, but it scores the same strict cohort as Ridge and persistence: complete target vectors marked `target_valid` whose full engineered predictor vectors are present. The other predictors therefore define comparability only; they are never passed to the model.

## Helper functions

The helpers validate the scoring cohort, independently verify each chronological physical water-level series, prepare a temporary interpolated training input, recompute calendar exogenous regressors for arbitrary timestamps, calculate metrics, and produce a prediction preview.

In [ ]:
def eligible_rows(
    frame: pd.DataFrame, *, station_id: str, artifact_name: str
) -> pd.Series:
    """Return model-ready rows and reject incomplete feature artifacts."""
    required_columns = {"timestamp", "target_valid", *FEATURE_COLUMNS, *TARGET_COLUMNS}
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )

    eligible = frame["target_valid"].eq(True) & frame[FEATURE_COLUMNS].notna().all(
        axis=1
    )
    if frame.loc[eligible, TARGET_COLUMNS].isna().any(axis=None):
        raise ValueError(
            f"{station_id} {artifact_name} artifact has null targets in target-valid rows"
        )
    return eligible

In [ ]:
def water_level_series(
    frame: pd.DataFrame, *, station_id: str, artifact_name: str
) -> np.ndarray:
    """Return one UTC-hourly station water-level series with no infinities."""
    required_columns = {"timestamp", "station_id", "water_level"}
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )
    if frame.empty:
        raise ValueError(f"{station_id} {artifact_name} artifact is empty")

    timestamp_dtype = frame["timestamp"].dtype
    if (
        not isinstance(timestamp_dtype, pd.DatetimeTZDtype)
        or str(timestamp_dtype.tz) != "UTC"
    ):
        raise ValueError(
            f"{station_id} {artifact_name} timestamps must be timezone-aware UTC"
        )
    timestamps = pd.DatetimeIndex(frame["timestamp"])
    if timestamps.hasnans:
        raise ValueError(f"{station_id} {artifact_name} timestamps must be complete")
    expected_grid = pd.date_range(timestamps[0], periods=len(timestamps), freq="h")
    if timestamps.has_duplicates or not timestamps.equals(expected_grid):
        raise ValueError(
            f"{station_id} {artifact_name} timestamps must be unique, ascending, and hourly"
        )

    station_ids = frame["station_id"].drop_duplicates().tolist()
    if station_ids != [station_id]:
        raise ValueError(
            f"{artifact_name} artifact must contain only station {station_id!r}; "
            f"got {station_ids!r}"
        )

    try:
        values = pd.to_numeric(frame["water_level"], errors="raise").to_numpy(
            dtype=float
        )
    except (TypeError, ValueError) as error:
        raise ValueError(
            f"{station_id} {artifact_name} water_level values must be numeric"
        ) from error
    if np.isinf(values).any():
        raise ValueError(
            f"{station_id} {artifact_name} water_level values must not contain infinities"
        )
    return values

In [ ]:
def interpolated_train_water_levels(values: np.ndarray) -> tuple[np.ndarray, int]:
    """Linearly fill internal training gaps for SARIMAX only."""
    missing_rows = int(np.isnan(values).sum())
    interpolated = pd.Series(values).interpolate(
        method=TRAIN_WATER_LEVEL_INTERPOLATION["method"],
        limit_area=TRAIN_WATER_LEVEL_INTERPOLATION["limit_area"],
    )
    prepared = interpolated.to_numpy(dtype=float)
    if not np.isfinite(prepared).all():
        raise ValueError(
            "SARIMAX training input has unfillable water_level gaps at a series boundary"
        )
    return prepared, missing_rows

In [ ]:
def calendar_exog(timestamps: pd.DatetimeIndex) -> pd.DataFrame:
    """Recompute the six calendar Fourier exog columns for arbitrary UTC timestamps."""
    converted = pd.DatetimeIndex(timestamps).tz_convert(CALENDAR_TIMEZONE)
    hour_angle = 2.0 * math.pi * converted.hour / 24.0
    weekday_angle = 2.0 * math.pi * converted.dayofweek / 7.0
    days_in_year = np.where(converted.is_leap_year, 366.0, 365.0)
    year_angle = 2.0 * math.pi * (converted.dayofyear - 1) / days_in_year
    return pd.DataFrame(
        {
            "utc_hour_sin": np.sin(hour_angle),
            "utc_hour_cos": np.cos(hour_angle),
            "utc_day_of_week_sin": np.sin(weekday_angle),
            "utc_day_of_week_cos": np.cos(weekday_angle),
            "utc_day_of_year_sin": np.sin(year_angle),
            "utc_day_of_year_cos": np.cos(year_angle),
        },
        index=timestamps,
    )

In [ ]:
def metric_tables(
    actual: pd.DataFrame, predictions: np.ndarray, *, station_id: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate aggregate and horizon-specific MAE/RMSE."""
    aggregate = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "scored_issue_times": len(actual),
                "scored_values": actual.size,
                "mae": mean_absolute_error(
                    actual.to_numpy().ravel(), predictions.ravel()
                ),
                "rmse": root_mean_squared_error(
                    actual.to_numpy().ravel(), predictions.ravel()
                ),
            }
        ]
    )
    per_horizon = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "horizon_hours": horizon,
                "target": target,
                "mae": mean_absolute_error(actual[target], predictions[:, horizon - 1]),
                "rmse": root_mean_squared_error(
                    actual[target], predictions[:, horizon - 1]
                ),
            }
            for horizon, target in enumerate(TARGET_COLUMNS, start=1)
        ]
    )
    return aggregate, per_horizon


def prediction_preview(frame: pd.DataFrame, predictions: np.ndarray) -> pd.DataFrame:
    """Return issue timestamps, actual targets, and direct multi-step predictions."""
    predicted = pd.DataFrame(
        predictions,
        columns=[f"prediction_{target}" for target in TARGET_COLUMNS],
        index=frame.index,
    )
    return pd.concat([frame[["timestamp", *TARGET_COLUMNS]], predicted], axis=1)

## Load and validate feature artifacts

Resolve the train/test parquet paths for the target station, failing fast if either is missing. Then validate each physical water-level series independently: it must belong only to the target station, use a contiguous UTC-hourly timeline, and contain numeric values without infinities. Short-gap values marked `imputed=True` and preserved long gaps remain valid physical-artifact values.

In [ ]:
station_id = TARGET_STATION_ID
train_path = PROCESSED_DIR / f"{station_id}_train_features.parquet"
test_path = PROCESSED_DIR / f"{station_id}_test_features.parquet"
for artifact_path in (train_path, test_path):
    if not artifact_path.is_file():
        raise FileNotFoundError(
            f"Missing feature artifact for {station_id}: {artifact_path}"
        )

train_features = pd.read_parquet(train_path)
test_features = pd.read_parquet(test_path)
train_water_levels = water_level_series(
    train_features, station_id=station_id, artifact_name="train"
)
test_water_levels = water_level_series(
    test_features, station_id=station_id, artifact_name="test"
)

## Verify calendar exog reconstruction

`get_forecast` needs exog values for future steps that are not always present as rows in the test artifact, so `calendar_exog()` recomputes them directly from timestamps. Before relying on it for those out-of-range steps, confirm it reproduces the existing per-row calendar columns for in-range timestamps to floating-point precision.

In [ ]:
reconstructed_test_exog = calendar_exog(pd.DatetimeIndex(test_features["timestamp"]))
if not np.allclose(
    reconstructed_test_exog[CALENDAR_EXOG_COLUMNS].to_numpy(),
    test_features[CALENDAR_EXOG_COLUMNS].to_numpy(),
):
    raise ValueError("calendar_exog() does not reproduce the artifact's calendar columns")

## Apply the eligibility cohort

Restrict metrics to the shared cohort and stop early if either feature split has no usable rows. The SARIMAX fit below uses a temporary interpolation of internal training gaps; it never rewrites the feature artifact.

In [ ]:
train_mask = eligible_rows(train_features, station_id=station_id, artifact_name="train")
test_mask = eligible_rows(test_features, station_id=station_id, artifact_name="test")
if not train_mask.any():
    raise ValueError(f"{station_id} train artifact has no eligible model rows")
if not test_mask.any():
    raise ValueError(f"{station_id} test artifact has no eligible scoring rows")

test_rows = test_features.loc[test_mask]
train_model_values, interpolated_train_rows = interpolated_train_water_levels(
    train_water_levels
)

## Fit the fixed-order SARIMAX model

Fits one SARIMAX model with a fixed, hand-picked order and a minimal seasonal term (P=1, D=0, Q=0, period 24), directly on the full train water-level series (linearly interpolating internal gaps only, matching Auto-ARIMA's training input) using the six calendar Fourier terms as exogenous regressors. No validation split, cross-validation, holdout observations, test rows, or other engineered predictors are used.

In [ ]:
train_exog = train_features[CALENDAR_EXOG_COLUMNS].to_numpy()

sarimax_model = SARIMAX(
    train_model_values,
    exog=train_exog,
    order=SARIMAX_ORDER,
    seasonal_order=SARIMAX_SEASONAL_ORDER,
)
sarimax_result = sarimax_model.fit(disp=False)

model_configuration = pd.DataFrame(
    [
        {
            "station_id": station_id,
            "model": "SARIMAX",
            "train_rows": len(train_water_levels),
            "interpolated_train_rows": interpolated_train_rows,
            "order": SARIMAX_ORDER,
            "seasonal_order": SARIMAX_SEASONAL_ORDER,
            "exog_columns": ", ".join(CALENDAR_EXOG_COLUMNS),
            "aic": sarimax_result.aic,
        }
    ]
)
print(f"SARIMAX fixed-order fit for {station_id}")
display(model_configuration)
display(pd.DataFrame([TRAIN_WATER_LEVEL_INTERPOLATION]))

## Issue rolling test forecasts

For every chronological test timestamp, first append its observed water level and matching calendar exog to the fitted SARIMAX state without refitting. A preserved missing value is appended as missing rather than interpolated, so no future test observation is used. Then issue one 24-step forecast using freshly computed calendar exog for the future steps, which may fall beyond the timestamps present in the test artifact. This keeps the current observation available whenever it exists while keeping the train-selected parameters fixed.

In [ ]:
state = sarimax_result
test_timestamps = pd.DatetimeIndex(test_features["timestamp"])
test_exog = test_features[CALENDAR_EXOG_COLUMNS].to_numpy()

all_test_predictions: list[np.ndarray] = []
for step, (timestamp, water_level) in enumerate(zip(test_timestamps, test_water_levels)):
    state = state.append([water_level], exog=test_exog[step : step + 1], refit=False)
    future_timestamps = pd.date_range(
        timestamp + pd.Timedelta(hours=1), periods=FORECAST_HORIZON_HOURS, freq="h"
    )
    future_exog = calendar_exog(future_timestamps).to_numpy()
    forecast = np.asarray(
        state.get_forecast(
            steps=FORECAST_HORIZON_HOURS, exog=future_exog
        ).predicted_mean,
        dtype=float,
    )
    if forecast.shape != (FORECAST_HORIZON_HOURS,):
        raise RuntimeError(
            f"Expected {FORECAST_HORIZON_HOURS} forecast values; got {forecast.shape}"
        )
    all_test_predictions.append(forecast)

test_predictions = np.vstack(all_test_predictions)[test_mask.to_numpy()]

## Evaluate on the test cohort

Build aggregate and per-horizon metric tables, and preview a few predicted rows against actuals.

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS], test_predictions, station_id=station_id
)
print(f"SARIMAX test results for {station_id}")
display(aggregate_metrics)
display(per_horizon_metrics)
display(prediction_preview(test_rows, test_predictions).head(PREDICTION_PREVIEW_ROWS))